# Leave-One-Out Evaluation Plots

This notebook generates the leave-one-out evaluation plots for Berlin, Santiago, and Yangon.
Each city has separate latency and throughput figures comparing Ground Truth, Full Model, and No-City Model.

The hourly CSV data must be present in `data/` (generated by the individual city evaluation scripts in the model-data-pipeline).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import rcParams
from matplotlib.ticker import MaxNLocator
from pathlib import Path

ROOT = Path('.').resolve()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

FONT_DIR = ROOT.parent / 'cmu-sans-serif' / 'Sans'
if FONT_DIR.exists():
    for font_file in FONT_DIR.glob('*.ttf'):
        fm.fontManager.addfont(str(font_file))

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['CMU Sans Serif']
plt.rcParams.update({
    'text.usetex': False,
    'font.size': 20,
    'pdf.fonttype': 42,
})

In [2]:
EVAL_START = '2025-11-24'
EVAL_END = '2025-11-30'
FULL_HOURS = pd.date_range(
    EVAL_START,
    pd.Timestamp(EVAL_END) + pd.Timedelta(hours=23),
    freq='h', tz='UTC',
)

colors = plt.get_cmap('tab10').colors
window_size = 5

CITIES = [
    {'key': 'berlin',   'label': 'Berlin'},
    {'key': 'santiago', 'label': 'Santiago'},
    {'key': 'yangon',   'label': 'Yangon'},
]

In [3]:
def load_hourly(csv_path):
    df = pd.read_csv(csv_path)
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    if 'noberlin' in df.columns:
        df = df.rename(columns={'noberlin': 'nocity'})
    df['timestamp'] = df['timestamp'].dt.floor('h')
    df = df.groupby('timestamp')[['gt', 'full', 'nocity']].mean().reset_index()
    df = df.set_index('timestamp').reindex(FULL_HOURS).reset_index()
    df = df.rename(columns={'index': 'timestamp'})
    return df


def midnight_positions():
    return [day * 24 for day in range(8)]


def eight_hour_positions():
    midnights = set(midnight_positions())
    all_pos, non_midnight = [], []
    for day in range(7):
        for h in (0, 8, 16):
            pos = day * 24 + h
            all_pos.append(pos)
            if pos not in midnights:
                non_midnight.append(pos)
    all_pos.append(7 * 24)
    return all_pos, non_midnight


def setup_xticks(ax):
    all_8h, non_midnight = eight_hour_positions()

    hour_labels = []
    for day in range(7):
        for h in (0, 8, 16):
            hour_labels.append(str(h))
    hour_labels.append('0')

    ax.set_xticks(all_8h)
    ax.set_xticklabels(hour_labels, fontsize=18)
    ax.tick_params(axis='x', which='major', length=4, width=0.8)

    for day_offset in range(7):
        ts = pd.Timestamp(EVAL_START, tz='UTC') + pd.Timedelta(days=day_offset)
        mid_x = day_offset * 24 + 12
        ax.text(mid_x, 1.02, ts.strftime('%a'), transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=20)

    for p in non_midnight:
        ax.axvline(x=p, color='gray', linewidth=0.5, linestyle='-', alpha=0.15)

    for mp in midnight_positions():
        ax.axvline(x=mp, color='black', linewidth=0.8, linestyle='-', alpha=0.5)

## Berlin

In [ ]:
city = CITIES[0]
key, label = city['key'], city['label']
df_lat = load_hourly(DATA_DIR / f'{key}_hourly_latency.csv')
df_tput = load_hourly(DATA_DIR / f'{key}_hourly_throughput.csv')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_lat))
ax.plot(x, df_lat['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x, df_lat['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x, df_lat['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('ms')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_latency.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_latency.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
x2 = np.arange(len(df_tput))
ax.plot(x2, df_tput['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x2, df_tput['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x2, df_tput['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('Mbps')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
ax.legend(frameon=True, facecolor='white', edgecolor='black', framealpha=0.9, loc='upper right', fontsize=14, handlelength=1.5, handletextpad=0.5, borderpad=0.4)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_throughput.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_throughput.png', dpi=300, bbox_inches='tight')
plt.show()

## Santiago

In [ ]:
city = CITIES[1]
key, label = city['key'], city['label']
df_lat = load_hourly(DATA_DIR / f'{key}_hourly_latency.csv')
df_tput = load_hourly(DATA_DIR / f'{key}_hourly_throughput.csv')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_lat))
ax.plot(x, df_lat['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x, df_lat['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x, df_lat['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('ms')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_latency.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_latency.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
x2 = np.arange(len(df_tput))
ax.plot(x2, df_tput['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x2, df_tput['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x2, df_tput['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('Mbps')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
ax.legend(frameon=True, facecolor='white', edgecolor='black', framealpha=0.9, loc='upper right', fontsize=14, handlelength=1.5, handletextpad=0.5, borderpad=0.4)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_throughput.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_throughput.png', dpi=300, bbox_inches='tight')
plt.show()

## Yangon

In [ ]:
city = CITIES[2]
key, label = city['key'], city['label']
df_lat = load_hourly(DATA_DIR / f'{key}_hourly_latency.csv')
df_tput = load_hourly(DATA_DIR / f'{key}_hourly_throughput.csv')

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_lat))
ax.plot(x, df_lat['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x, df_lat['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x, df_lat['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('ms')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_latency.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_latency.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
x2 = np.arange(len(df_tput))
ax.plot(x2, df_tput['gt'].interpolate().rolling(window=window_size, center=True, min_periods=1).mean(), '-', color=colors[0], linewidth=2, label='Ground Truth')
ax.plot(x2, df_tput['full'].rolling(window=window_size, center=True, min_periods=1).mean(), '--', color=colors[1], linewidth=2, label='Full Model')
ax.plot(x2, df_tput['nocity'].rolling(window=window_size, center=True, min_periods=1).mean(), ':', color=colors[2], linewidth=2.5, label=f'No-{label} Model')
ax.set_ylabel('Mbps')
ax.yaxis.set_major_locator(MaxNLocator(nbins=6, integer=False))
setup_xticks(ax)
ax.legend(frameon=True, facecolor='white', edgecolor='black', framealpha=0.9, loc='upper right', fontsize=14, handlelength=1.5, handletextpad=0.5, borderpad=0.4)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / f'{key}_throughput.pdf', bbox_inches='tight', format='pdf')
fig.savefig(OUTPUT_DIR / f'{key}_throughput.png', dpi=300, bbox_inches='tight')
plt.show()